# Shareability Metric

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
from ssl_shareability_metric.ssl_encoder_shareability import ssl_shareability

In [2]:
df = pd.read_csv(Path().cwd().resolve().parent / "data/sample_data.csv")

## Collapse seperate date times into 1 date time feature

In [3]:
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

## Sort by date time to ensure t and t+1 relationship

In [4]:
df = df.sort_values("datetime", ascending=True).reset_index(drop=True)

## Drop non continous features to preserve a clean cross covariance matrix M

In [5]:
df = df.drop(columns=["No", "year", "month", "day", "hour", "wd", "station", "RAIN"])

## Handle NaN values

In [6]:
df = df.dropna().reset_index(drop=True)

## Create lagged pairs

In [7]:
current = df.copy()
future = df.shift(-1)

valid_pair = (df["datetime"].shift(-1) - df["datetime"]).eq(pd.Timedelta(hours=1))

x_current = current.loc[valid_pair].reset_index(drop=True)
x_future = future.loc[valid_pair].reset_index(drop=True)

x_current = x_current.drop(columns=["datetime"])
x_future = x_future.drop(columns=["datetime"])

x_current.head()

,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,WSPM
0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,4.4
1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,4.7
2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,5.6
3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,3.1
4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,2.0


## Convert to numpy for easier tensor work

In [8]:
current_np = np.array(x_current)
future_np = np.array(x_future)

print(current_np.shape)
print(future_np.shape)

(30943, 10)
(30943, 10)


## Establish train val and test splits

In [9]:
train_pct = 0.70
val_pct = 0.10

train_current = current_np[:int(train_pct * current_np.shape[0])]
train_future = future_np[:int(train_pct * future_np.shape[0])]

val_current = current_np[int(train_pct * current_np.shape[0]):int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0])]
val_future = future_np[int(train_pct * future_np.shape[0]):int(train_pct * future_np.shape[0]) + int(val_pct * future_np.shape[0])]

test_current = current_np[int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0]):]
test_future = future_np[int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0]):]

print(train_current.shape)
print(train_future.shape)

print(val_current.shape)
print(val_future.shape)

print(test_current.shape)
print(test_future.shape)

(21660, 10)
(21660, 10)
(3094, 10)
(3094, 10)
(6189, 10)
(6189, 10)


## Shareability score

In [10]:
shareability, shared, seperate = ssl_shareability(train_current, train_future)
print(f"Shareability Score: {shareability}, Shared: {shared}, Seperate: {seperate}")

Shareability Score: 0.9999427343409215, Shared: 0.998177335180484, Seperate: 0.9982344997370264
